# Which model should decide UnderCover's claims?

**The challenge:** find the cheapest model that gets at least 15 of 25 claims right, and does it **consistently** (15 or more on every run).

**What I did:** shortlisted cheap, tool-capable models, smoke-tested them on 5 claims, then ran the best ones on the full 25-claim queue several times each. I also tested one idea: giving the model UnderCover's **claims manual**, the handlers' internal rules, which the policy wording alone doesn't contain.

### The short answer

- **The claims manual was the biggest improvement I found.** It added about 4 points for DeepSeek and about 6 for GLM, for under a cent more per run. For Luna it added about 1, within the run-to-run noise, but it lifted Luna's worst run from 13 to 15.
- **Most accurate, and best value if accuracy matters:** DeepSeek V4.1 Flash with the manual, at 22 of 25. Compared with GLM, each extra correct claim costs about 0.3 cents, which is cheap if a wrong decision costs a handler's time or a wrongly paid claim.
- **Cheapest that clears 15 every time:** Luna with the manual as a single call, at 16 of 25 and $0.005 for the whole queue, but only just, since its worst run was exactly 15. GLM 5.3 Flash with the manual is next at $0.014, with more room (worst run 17).
- **The agent harness gave no reliable accuracy gain** (-1.2 for DeepSeek, +0.3 for GLM, +1.7 for Luna, all within the noise) **and always cost more**, from 1.2 to 2.4 times.

### Every setup I tested

Scores are claims right out of 25. Cost and time are for one run of the whole queue. "Single call" means one request with everything in the prompt, and "agent" means the model looks up the policy itself using tools.

| Setup | Runs | Score on each run | Mean | 15+ every run? | Cost per run | Time per run |
|---|---|---|---|---|---|---|
| DeepSeek V4.1 Flash, wording only | 6 | 18, 19, 19, 19, 16, 18 | 18.2 | Yes | $0.020 | 3.3 min |
| MiMo V2.6 Pro, wording only | 2 | 18, 17 | 17.5 | Yes | $0.042 | 13.2 min |
| GLM 5.3 Flash, wording only | 3 | 12, 13, 12 | 12.3 | **No** | $0.007 | 1.3 min |
| GLM 5.3 Flash + claims manual | 3 | 18, 19, 17 | 18.0 | Yes | $0.014 | 1.3 min |
| DeepSeek + claims manual, single call | 4 | 21, 23, 22, 22 | **22.0** | Yes | $0.027 | 3.4 min |
| DeepSeek + claims manual, agent | 6 | 21, 23, 20, 20, 20, 21 | 20.8 | Yes | $0.033 | 3.3 min |
| GLM 5.3 Flash + claims manual, agent | 3 | 17, 20, 18 | 18.3 | Yes | $0.029 | 2.7 min |
| Luna, wording only | 3 | 18, 15, 13 | 15.3 | **No** | $0.003 | 1.2 min |
| Luna + claims manual, single call | 3 | 17, 15, 16 | 16.0 | Yes, just | $0.005 | 1.3 min |
| Luna + claims manual, agent | 3 | 17, 15, 21 | 17.7 | Yes, just | $0.011 | 3.1 min |

Luna clears 15 on its worst run by exactly 0 points, so treat its "Yes" as fragile.

Below: the findings and recommendation, [what surprised me](#what-surprised-me), then the method, the caveats and a guide to re-running the experiment.

## Findings and recommendation

### Findings

1. **The claims manual helped every model, by very different amounts.** It raised DeepSeek from 18.2 to 22.0 and GLM from 12.3 to 18.0. Luna gained only about 1 point (15.3 to 16.0, within the noise), but its worst run improved from 13 to 15. For all three, the "manual" claims went from about 0 to about 3 of 5.
2. **A single call is as good as an agent here, and cheaper.** The agent gave no reliable accuracy gain for any of the three models (-1.2, +0.3 and +1.7 points, all within the noise), and it cost 1.2 to 2.4 times as much.
3. **A pricier model didn't help.** MiMo V2.6 Pro cost twice as much as DeepSeek and ran four times slower, with no gain on the same prompt.
4. **With the manual, every miss was a hard or "manual" claim.** DeepSeek got all 15 routine claims right, and missed about 3 of the other 10.

### Recommendation

1. **Put the claims manual in the prompt**, whichever model you choose.
2. **Decide claims with a single call**, not an agent, until an agent shows it can beat that.
3. **Choose the model by the cost of a mistake:**
   - **DeepSeek V4.1 Flash**, if accuracy matters: 22 of 25 at about a tenth of a cent per claim.
   - **Luna**, if the aim is the cheapest model that clears the bar: 16 of 25 at about 0.02 of a cent per claim. Its worst run was exactly 15, so it's the most fragile choice.
   - **GLM 5.3 Flash**, for a cheap option with more margin: 18 of 25 at about 0.06 of a cent per claim, worst run 17.
4. **Keep a handler on the hard and "manual" claims.** Even the best setup still misses some of them.

**How sure am I?** Confident that the manual helps, because both models improved and their score ranges don't overlap. Moderately sure that a single call beats an agent, because there are only two models and the differences are small. Least sure about Luna: it clears 15 by 0 points on its worst run, so three runs aren't enough to call it consistent. I also didn't test MiMo with the manual.

## What surprised me

1. **Without the manual, no model got a single "manual" claim right.** That is 0 of 5 in all 11 wording-only runs, so the best possible score without it is 20 of 25. The ceiling comes from missing information, not from model quality.
2. **The manual beat a model upgrade.** It lifted DeepSeek from 18.2 to 22.0 and GLM from 12.3 to 18.0, and the per-run ranges don't overlap for either model. Switching to the pricier MiMo moved the score by -0.7.
3. **The cheapest setup isn't the cheapest answer.** GLM without the manual cost only $0.007 a run, and never reached 15. With the manual it cost twice as much and passed every time.
4. **One setup can score anywhere from 16 to 19.** The same model, prompt and claims gave different scores across six runs. A single run can mislead, which is why "consistently" matters.
5. **The agent harness added cost, and no reliable accuracy.** The score changes were -1.2 for DeepSeek, +0.3 for GLM and +1.7 for Luna (helped by one 21 of 25 run), and the cost went up by 1.2, 2.0 and 2.4 times. All three accuracy differences are within the noise, so I read it as "no reliable accuracy benefit and a real cost penalty".
6. **MiMo cost twice as much, ran four times slower and was no more accurate on the same prompt.** I didn't test it with the manual, so this says nothing about MiMo's best case.
7. **Two claims stayed wrong even with the manual.** CLM-1019 and CLM-1024 were missed in all 4 DeepSeek single-call runs with the manual. They're worth a look by hand.
8. **The cheapest model needed the manual least, and still came out cheapest.** Luna scored 15.3 without the manual and 16.0 with it, and the ranges overlap. What the manual did was lift its worst run from 13 to 15, which is what put it over the "consistent" bar. At $0.005 a run it's the cheapest setup that cleared 15 every time, by the smallest possible margin.

<details>
<summary><b>Method: how I ran it</b></summary>

**The data.** 25 claims from the lab, each with the handler's real decision and payout. 15 are routine, 5 are hard and 5 are "manual" claims, which need the claims manual or the customer's history to decide.

**Scoring.** A claim counts as right when the model's decision matches the handler's **and** its payout is within $1.

**The prompt.** The system prompt has the instructions and the policy wording. The "with manual" runs also include `claims_manual.md`. In the single-call setup the policy record comes with the claim. In the agent setup the model fetches it with a `get_policy` tool and records its answer with a `record_decision` tool.

**Choosing the models.**
1. I listed the cheapest tool-capable models on OpenRouter and compared them with benchmarks from Artificial Analysis.
2. I smoke-tested GLM 5.3 Flash, MiMo V2.6 Pro, DeepSeek V4.1 Flash, Solar Mini4 and Mercury 2.5, with Luna as the baseline, on 5 claims at low reasoning effort. Solar was the most expensive and slowest, so I dropped it. DeepSeek had the best mix of accuracy, cost and speed. MiMo was the most accurate but about 5 times slower and 3.5 times pricier.
3. I ran DeepSeek and MiMo on the full queue, then added GLM later for comparison.

**Repeats.** Each setup ran several times. I define "consistent" as 15 or more on **every** run.

**Two sessions.** I ran the experiment in two sessions and pooled the results, because the spread between sessions is itself informative. That's why the run counts differ between rows.

</details>

<details>
<summary><b>Caveats and what I didn't test</b></summary>

**How far to trust the numbers**
- It's one queue of 25 claims, with 2 to 6 runs per setup, and one handler's decisions as the answer key.
- Run counts are uneven. DeepSeek has the most runs and MiMo the fewest, so MiMo's mean is the weakest.
- Prices and model behaviour change, and costs are what OpenRouter reported when I ran each setup.
- I haven't investigated why the agent adds cost without a reliable accuracy gain.

**The agent question**
- The challenge asks about the **agent**. I tested DeepSeek and GLM as agents, both with the manual.
- Luna in the agent with the manual is the cheapest agent that cleared 15 on every run ($0.011 a run), but by 0 points on its worst run. The lab's own wording-only Luna agent run scored 17 of 25 for $0.0086, and I didn't repeat it.

**Not tested**
- MiMo with the claims manual
- MiMo as an agent
- Prompt changes other than adding the manual
- Higher reasoning effort

**Next experiments:** MiMo with the manual, more Luna runs to see whether 15 holds, more runs per setup, and a hand review of CLM-1019 and CLM-1024.

</details>

<details>
<summary><b>Re-running it: what costs money and how long it takes</b></summary>

You need `OPENROUTER_API_KEY` in your `.env`. The shortlisting cell also uses a free `ARTIFICIAL_ANALYSIS_API_KEY`. Run the cells in order, because later cells use variables from earlier ones. Estimates come from my saved runs.

| Step | Time | Cost |
|---|---|---|
| Setup and the Luna baseline | about 1 min | about $0.004 |
| Smoke test (5 claims per model) | a few minutes | a few cents |
| DeepSeek (3 runs) and MiMo (1 run), wording only | about 23 min | about $0.10 |
| DeepSeek + manual, single call (3 runs) | about 10 min | about $0.08 |
| DeepSeek + manual, agent (3 runs) | about 10 min | about $0.10 |
| GLM, wording only (3 runs) | about 4 min | about $0.02 |
| GLM + manual (3 runs) | about 4 min | about $0.04 |
| GLM + manual, agent (3 runs) | about 8 min | about $0.09 |
| Luna, three setups (9 runs) | about 17 min | about $0.06 |
| **Everything** | **about 75 min** | **about $0.50** |

To go faster, lower the number of runs in each loop to 1, or skip the MiMo run, which is the slowest at about 13 minutes.

</details>

---

## Setup

The cells below are from Ed's lab, kept because the experiment uses them.

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field
import time

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

In [ ]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = False

DATA = next(
    folder / "data"
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "data" / "claims.json").is_file()
)
claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

In [ ]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

In [ ]:
def decide(claim, model, effort):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "effort": effort,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 1),
    }

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, effort, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [ ]:
luna_low = run("openai/gpt-6-luna", "low", claims, labels)

In [ ]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [ ]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.
{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

In [ ]:
import httpx


def cheapest_tool_models(count=25):
    """List the cheapest paid OpenRouter models that support tools, by price per million input tokens."""
    models = httpx.get(f"{OPENROUTER_BASE_URL}/models").json()["data"]
    rows = [
        {
            "model": m["id"],
            "input $/M": float(m["pricing"]["prompt"]) * 1_000_000,
            "output $/M": float(m["pricing"]["completion"]) * 1_000_000,
        }
        for m in models
        if "tools" in m["supported_parameters"] and float(m["pricing"]["prompt"]) > 0
    ]
    return pd.DataFrame(rows).sort_values("input $/M").head(count).reset_index(drop=True)

# Pull Models from Artificial Analysis To Search for Candidates

The call below calls the Artificial Analysis API and compares my proposed models against other models I may not have considered. 

In [ ]:
# Benchmarks from Artificial Analysis (https://artificialanalysis.ai/), to shortlist models before spending on our own evals.
# Needs ARTIFICIAL_ANALYSIS_API_KEY in .env: a free key from their Insights Platform, 1,000 requests a day.
import re

import httpx

AA_URL = "https://artificialanalysis.ai/api/v2/data/llms/models"
MY_PICKS = ["openai/gpt-6-luna", "meta/muse-spark-1.3", "z-ai/glm-5.3-flash", "moonshotai/kimi-k3", "xiaomi/mimo-v2.6-pro"]


def squash(text):
    """Lowercase and keep only letters and digits, so 'glm-5.3-flash' matches 'glm-5-3-flash'."""
    return re.sub(r"[^a-z0-9]", "", text.lower())


def fetch_benchmarks():
    """Every model Artificial Analysis has scored, with its intelligence, price and speed."""
    load_dotenv(override=True)
    response = httpx.get(AA_URL, headers={"x-api-key": os.getenv("ARTIFICIAL_ANALYSIS_API_KEY")}, timeout=30)
    response.raise_for_status()
    return response.json()["data"]


def benchmark_table(candidates, scored):
    """One row per Artificial Analysis entry whose slug contains a candidate's name. A model can match a few variants."""
    rows = []
    for candidate in candidates:
        name = squash(candidate.split("/")[1])
        for model in scored:
            if name in squash(model["slug"]):
                rows.append({
                    "openrouter": candidate,
                    "benchmarked as": model["name"],
                    "intelligence": model["evaluations"].get("artificial_analysis_intelligence_index"),
                    "blended $/M": model["pricing"].get("price_1m_blended_3_to_1"),
                    "tokens/s": model.get("median_output_tokens_per_second"),
                    "first token s": model.get("median_time_to_first_token_seconds"),
                })
    return pd.DataFrame(rows)


cheap = {model.removeprefix("~").removesuffix(":batch") for model in cheapest_tool_models()["model"]}
candidates = sorted(cheap | set(MY_PICKS))
benchmarks = benchmark_table(candidates, fetch_benchmarks())
print("No benchmark found for:", ", ".join(sorted(set(candidates) - set(benchmarks["openrouter"]))))

# TODO(human): rank `benchmarks` to pick the 5 models to smoke-test.

benchmarks

# Smoke Test The Candidates With 5 Claims

Based on the data, I chose to test GLM 5.3 Flash, Mimo V2.6 Pro, Deepseek 4.1 Flash, Solar Mini4 and Mercury 2.5 all on low reasoning effort. 

In [ ]:
# Smoke test: each shortlisted model on the 5 tiny claims, with Luna at low as the baseline on the same claims.
SHORTLIST = [
    ("openai/gpt-6-luna", "low"),
    ("z-ai/glm-5.3-flash", "low"),
    ("xiaomi/mimo-v2.6-pro", "low"),
    ("deepseek/deepseek-v4.1-flash", "low"),
    ("upstage/solar-mini4", "low"),
    ("inception/mercury-2.5", "low"),
]

tiny_claims, tiny_labels = get_sample_data(tiny=True)
smoke = {}
for model, effort in SHORTLIST:
    try:
        smoke[(model, effort)] = run(model, effort, tiny_claims, tiny_labels)
    except Exception as error:
        print(f"{model} at {effort}: failed with {type(error).__name__}: {error}")

summary = pd.DataFrame([
    {
        "model": model,
        "effort": effort,
        "right": sum(result["right"] for result in results),
        "of": len(results),
        "$ per claim": sum(result["cost"] for result in results) / len(results),
        "seconds per claim": sum(result["seconds"] for result in results) / len(results),
    }
    for (model, effort), results in smoke.items()
])
summary["$ for 25 claims"] = summary["$ per claim"] * 25
summary.sort_values("$ per claim")


Luna, GLM Flash and Solar Mini4 performed roughly the same. Solar was the most expensive and slowest so that one is ruled out.

Deepseek overall had the best mix of accuracy, cost and speed. 

Mimo was the most accurate but at the tradeoff of 5x the latency and 3.5x the cost. 

In [ ]:
def summarize(experiments):
    """One row per (model, effort): right answers, cost and time per claim, and cost projected to 25 claims."""
    summary = pd.DataFrame([
        {
            "model": model,
            "effort": effort,
            "right": sum(result["right"] for result in results),
            "of": len(results),
            "$ per claim": sum(result["cost"] for result in results) / len(results),
            "seconds per claim": sum(result["seconds"] for result in results) / len(results),
        }
        for (model, effort), results in experiments.items()
    ])
    summary["$ for 25 claims"] = summary["$ per claim"] * 25
    return summary.sort_values("$ per claim")


summarize(smoke)


Based on these findings, ran full tests with Deepseek and Mimo.

In [ ]:
# Full runs on the 25-claim queue. MiMo runs once first: about 14 minutes and about 15x Luna's cost.
FULL_RUNS = [
    ("deepseek/deepseek-v4.1-flash", "low", 3),
    ("xiaomi/mimo-v2.6-pro", "low", 1),
]
RESULTS_FILE = Path("results/week1day2_full_runs.jsonl")
RESULTS_FILE.parent.mkdir(exist_ok=True)

full = {("openai/gpt-6-luna", "low"): [luna_low]}
for model, effort, times in FULL_RUNS:
    for attempt in range(1, times + 1):
        results = run(model, effort, claims, labels)
        full.setdefault((model, effort), []).append(results)
        with RESULTS_FILE.open("a") as file:
            file.writelines(json.dumps({**result, "run": attempt}) + "\n" for result in results)


In [ ]:
def summarize_runs(experiments):
    """One row per (model, effort): the score on each run, and average cost and time over the runs."""
    rows = []
    for (model, effort), runs in experiments.items():
        scores = [sum(result["right"] for result in results) for results in runs]
        every = [result for results in runs for result in results]
        rows.append({
            "model": model,
            "effort": effort,
            "scores": scores,
            "15+ every run": min(scores) >= 15,
            "$ per 25 claims": sum(result["cost"] for result in every) / len(runs),
            "seconds per claim": sum(result["seconds"] for result in every) / len(every),
        })
    return pd.DataFrame(rows).sort_values("$ per 25 claims")


summarize_runs(full)


In [ ]:
def confusion(results):
    """Handler's decision (rows) against the model's (columns), over all the results given."""
    return pd.crosstab(
        pd.Series([labels[result["claim_id"]]["decision"] for result in results], name="handler"),
        pd.Series([result["decision"] for result in results], name="model"),
    )


In [ ]:
# Experiment: does adding the claims manual fix the manual claims? One DeepSeek run, about 3 minutes and about $0.04.
claims_manual = (DATA / "claims_manual.md").read_text(encoding="utf-8")
SYSTEM_PROMPT_WORDING_ONLY = SYSTEM_PROMPT
SYSTEM_PROMPT_WITH_MANUAL = f"""{INSTRUCTIONS}
Where the claims manual below sets a rule, follow it.

{policy_wording}

# Claims manual

{claims_manual}
"""

# make_messages reads SYSTEM_PROMPT when it's called, so swap it for this run and always put it back.
SYSTEM_PROMPT = SYSTEM_PROMPT_WITH_MANUAL
try:
    with_manual = run("deepseek/deepseek-v4.1-flash", "low", claims, labels)
finally:
    SYSTEM_PROMPT = SYSTEM_PROMPT_WORDING_ONLY

full[("deepseek/deepseek-v4.1-flash + manual", "low")] = [with_manual]
display(summarize_runs(full))

pd.DataFrame([
    {
        "claim": result["claim_id"],
        "needs": labels[result["claim_id"]]["needs"],
        "handler": labels[result["claim_id"]]["decision"],
        "with manual": result["decision"],
        "right": result["right"],
    }
    for result in with_manual
    if result["difficulty"] == "manual"
])


In [ ]:
before = {}
for results in full[("deepseek/deepseek-v4.1-flash", "low")]:
    for result in results:
        before[result["claim_id"]] = before.get(result["claim_id"], 0) + result["right"]

changes = pd.DataFrame([
    {
        "claim": result["claim_id"],
        "difficulty": result["difficulty"],
        "needs": labels[result["claim_id"]]["needs"],
        "right before (of 3)": before[result["claim_id"]],
        "right with manual": result["right"],
    }
    for result in with_manual
])
changes[changes["right before (of 3)"] != 3 * changes["right with manual"]]


In [ ]:
# Two more DeepSeek runs with the claims manual, to see if 22 holds.
MANUAL_KEY = ("deepseek/deepseek-v4.1-flash + manual", "low")

SYSTEM_PROMPT = SYSTEM_PROMPT_WITH_MANUAL
try:
    for attempt in (2, 3):
        results = run("deepseek/deepseek-v4.1-flash", "low", claims, labels)
        full.setdefault(MANUAL_KEY, []).append(results)
        with RESULTS_FILE.open("a") as file:
            file.writelines(json.dumps({**result, "run": attempt, "prompt": "with manual"}) + "\n" for result in results)
finally:
    SYSTEM_PROMPT = SYSTEM_PROMPT_WORDING_ONLY

summarize_runs(full)


In [ ]:
# Three GLM runs with the wording only, as the baseline for the claims manual runs below.
assert SYSTEM_PROMPT == SYSTEM_PROMPT_WORDING_ONLY

for attempt in (1, 2, 3):
    results = run("z-ai/glm-5.3-flash", "low", claims, labels)
    full.setdefault(("z-ai/glm-5.3-flash", "low"), []).append(results)
    with RESULTS_FILE.open("a") as file:
        file.writelines(json.dumps({**result, "run": attempt}) + "\n" for result in results)

summarize_runs(full)


In [ ]:
# Three GLM runs with the claims manual for benchmarking
MANUAL_KEY = ("z-ai/glm-5.3-flash + manual", "low")

SYSTEM_PROMPT = SYSTEM_PROMPT_WITH_MANUAL
try:
    for attempt in (1, 2, 3):
        results = run("z-ai/glm-5.3-flash", "low", claims, labels)
        full.setdefault(MANUAL_KEY, []).append(results)
        with RESULTS_FILE.open("a") as file:
            file.writelines(json.dumps({**result, "run": attempt, "prompt": "with manual"}) + "\n" for result in results)
finally:
    SYSTEM_PROMPT = SYSTEM_PROMPT_WORDING_ONLY

summarize_runs(full)


In [ ]:
def right_counts(runs):
    """How many of the runs got each claim right."""
    counts = {}
    for results in runs:
        for result in results:
            counts[result["claim_id"]] = counts.get(result["claim_id"], 0) + result["right"]
    return counts


before = right_counts(full[("deepseek/deepseek-v4.1-flash", "low")])
after = right_counts(full[MANUAL_KEY])
changes = pd.DataFrame([
    {
        "claim": claim_id,
        "difficulty": labels[claim_id]["difficulty"],
        "needs": labels[claim_id]["needs"],
        "without manual (of 3)": before[claim_id],
        "with manual (of 3)": after[claim_id],
        "tests": labels[claim_id]["tests"],
    }
    for claim_id in before
    if before[claim_id] != after[claim_id]
])
changes


In [ ]:
AGENT_SYSTEM_PROMPT_WITH_MANUAL = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.
Where the claims manual below sets a rule, follow it.

{policy_wording}

# Claims manual

{claims_manual}
"""


def make_agent(model, effort, system_prompt):
    """A claims agent on any OpenRouter model, with the policy lookup and the decision recorder."""
    llm = ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort=effort, timeout=120, max_retries=5)
    return create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=system_prompt)


def run_agent(model, effort, system_prompt, claims, labels):
    """Run a fresh agent on the claims and compare what it recorded against the labels."""
    agent = make_agent(model, effort, system_prompt)
    recorded.clear()
    results = []
    for claim in claims:
        start = time.time()
        result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
        answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
        label = labels[claim["claim_id"]]
        results.append({
            "model": model,
            "effort": effort,
            "technique": "Agent",
            "claim_id": claim["claim_id"],
            **answer,
            "difficulty": label["difficulty"],
            "right": is_right(answer, label),
            "cost": sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai"),
            "seconds": round(time.time() - start, 1),
        })
    right = sum(result["right"] for result in results)
    print(f"{model} agent at {effort}: {right} of {len(results)} right, {len(recorded)} recorded, for ${sum(result['cost'] for result in results):.4f}")
    return results


AGENT_KEY = ("deepseek/deepseek-v4.1-flash + manual, agent", "low")
for attempt in (1, 2, 3):
    results = run_agent("deepseek/deepseek-v4.1-flash", "low", AGENT_SYSTEM_PROMPT_WITH_MANUAL, claims, labels)
    full.setdefault(AGENT_KEY, []).append(results)
    with RESULTS_FILE.open("a") as file:
        file.writelines(json.dumps({**result, "run": attempt, "prompt": "with manual"}) + "\n" for result in results)

summarize_runs(full)


In [ ]:
# Three GLM runs inside the agent harness with the claims manual, to compare with its single call and with DeepSeek's agent.
GLM_AGENT_KEY = ("z-ai/glm-5.3-flash + manual, agent", "low")
for attempt in (1, 2, 3):
    results = run_agent("z-ai/glm-5.3-flash", "low", AGENT_SYSTEM_PROMPT_WITH_MANUAL, claims, labels)
    full.setdefault(GLM_AGENT_KEY, []).append(results)
    with RESULTS_FILE.open("a") as file:
        file.writelines(json.dumps({**result, "run": attempt, "prompt": "with manual"}) + "\n" for result in results)

summarize_runs(full)

In [ ]:
# Luna in three setups, three runs each: single call with the wording only, single call with the claims manual, and the agent with the claims manual.
# The first setup also holds the lab's own luna_low run, so it ends up with four runs.
LUNA = ("openai/gpt-6-luna", "low")
SETUPS = [
    # (key in full, system prompt, with the manual?, agent?)
    (LUNA, SYSTEM_PROMPT_WORDING_ONLY, False, False),
    (("openai/gpt-6-luna + manual", "low"), SYSTEM_PROMPT_WITH_MANUAL, True, False),
    (("openai/gpt-6-luna + manual, agent", "low"), AGENT_SYSTEM_PROMPT_WITH_MANUAL, True, True),
]


def run_luna(system_prompt, agent):
    """One Luna run on the whole queue: the agent, or a single call with SYSTEM_PROMPT swapped in and restored after."""
    global SYSTEM_PROMPT
    if agent:
        return run_agent(*LUNA, system_prompt, claims, labels)
    SYSTEM_PROMPT = system_prompt
    try:
        return run(*LUNA, claims, labels)
    finally:
        SYSTEM_PROMPT = SYSTEM_PROMPT_WORDING_ONLY


for key, system_prompt, with_manual, agent in SETUPS:
    for attempt in (1, 2, 3):
        results = run_luna(system_prompt, agent)
        full.setdefault(key, []).append(results)
        prompt = "with manual" if with_manual else "wording only"
        with RESULTS_FILE.open("a") as file:
            file.writelines(json.dumps({**result, "run": attempt, "prompt": prompt}) + "\n" for result in results)

summarize_runs(full)